In [1]:
import pandas as pd

# 1. Load data

In [2]:
data = pd.read_csv("../data/raw/award_winners.csv")
data.head()

,key_id,tournament_id,tournament_name,award_id,award_name,shared,player_id,family_name,given_name,team_id,team_name,team_code
0,1,WC-1930,1930 FIFA Men's World Cup,A-4,Golden Boot,0,P-56486,Stábile,Guillermo,T-03,Argentina,ARG
1,2,WC-1930,1930 FIFA Men's World Cup,A-5,Silver Boot,0,P-18628,Cea,Pedro,T-84,Uruguay,URY
2,3,WC-1930,1930 FIFA Men's World Cup,A-6,Bronze Boot,0,P-71973,Patenaude,Bert,T-83,United States,USA
3,4,WC-1934,1934 FIFA Men's World Cup,A-4,Golden Boot,0,P-60513,Nejedlý,Oldřich,T-21,Czechoslovakia,CSK
4,5,WC-1934,1934 FIFA Men's World Cup,A-5,Silver Boot,1,P-77348,Conen,Edmund,T-31,Germany,DEU


# 2. Data analysis

In [3]:
data.columns

Index(['key_id', 'tournament_id', 'tournament_name', 'award_id', 'award_name',
       'shared', 'player_id', 'family_name', 'given_name', 'team_id',
       'team_name', 'team_code'],
      dtype='str')

In [4]:
data.isna().sum()

key_id             0
tournament_id      0
tournament_name    0
award_id           0
award_name         0
shared             0
player_id          0
family_name        0
given_name         0
team_id            0
team_name          0
team_code          0
dtype: int64

In [5]:
data.duplicated().sum()

np.int64(0)

# 3. Extract year

In [6]:
def extract_year(row: pd.DataFrame) -> str:
    return row["tournament_id"][3:]

years = data.apply(extract_year, axis=1, result_type="expand")
data["Year"] = years
data[["tournament_id", "Year"]]

,tournament_id,Year
0,WC-1930,1930
1,WC-1930,1930
2,WC-1930,1930
3,WC-1934,1934
4,WC-1934,1934
...,...,...
195,WC-2022,2022
196,WC-2022,2022
197,WC-2022,2022
198,WC-2022,2022


# 4. Extract Player Name

In [8]:
data["given_name"].value_counts()

given_name
not applicable    28
Diego              4
Thomas             4
Teófilo            3
Paolo              3
                  ..
Ellen              1
Sari               1
Giulia             1
Olivier            1
Emiliano           1
Name: count, Length: 124, dtype: int64

In [9]:
def extract_player_name(row: pd.DataFrame) -> str:
    first_name = row["given_name"].strip()
    last_name = row["family_name"].strip()

    if first_name == "not applicable":
        return last_name
    return f"{first_name} {last_name}"


In [10]:
player_names = data.apply(extract_player_name, axis=1, result_type="expand")
data["Player Name"] = player_names
data[["given_name", "family_name", "Player Name"]]

,given_name,family_name,Player Name
0,Guillermo,Stábile,Guillermo Stábile
1,Pedro,Cea,Pedro Cea
2,Bert,Patenaude,Bert Patenaude
3,Oldřich,Nejedlý,Oldřich Nejedlý
4,Edmund,Conen,Edmund Conen
...,...,...,...
195,Kylian,Mbappé,Kylian Mbappé
196,Lionel,Messi,Lionel Messi
197,Olivier,Giroud,Olivier Giroud
198,Emiliano,Martínez,Emiliano Martínez


# 5. Cleaning

## Remove female world cups

In [11]:
data = data[~data["tournament_name"].str.contains("Women")].copy()
data

,key_id,tournament_id,tournament_name,award_id,award_name,shared,player_id,family_name,given_name,team_id,team_name,team_code,Year,Player Name
0,1,WC-1930,1930 FIFA Men's World Cup,A-4,Golden Boot,0,P-56486,Stábile,Guillermo,T-03,Argentina,ARG,1930,Guillermo Stábile
1,2,WC-1930,1930 FIFA Men's World Cup,A-5,Silver Boot,0,P-18628,Cea,Pedro,T-84,Uruguay,URY,1930,Pedro Cea
2,3,WC-1930,1930 FIFA Men's World Cup,A-6,Bronze Boot,0,P-71973,Patenaude,Bert,T-83,United States,USA,1930,Bert Patenaude
3,4,WC-1934,1934 FIFA Men's World Cup,A-4,Golden Boot,0,P-60513,Nejedlý,Oldřich,T-21,Czechoslovakia,CSK,1934,Oldřich Nejedlý
4,5,WC-1934,1934 FIFA Men's World Cup,A-5,Silver Boot,1,P-77348,Conen,Edmund,T-31,Germany,DEU,1934,Edmund Conen
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,WC-2022,2022 FIFA Men's World Cup,A-4,Golden Boot,0,P-64077,Mbappé,Kylian,T-30,France,FRA,2022,Kylian Mbappé
196,197,WC-2022,2022 FIFA Men's World Cup,A-5,Silver Boot,0,P-14758,Messi,Lionel,T-03,Argentina,ARG,2022,Lionel Messi
197,198,WC-2022,2022 FIFA Men's World Cup,A-6,Bronze Boot,0,P-89750,Giroud,Olivier,T-30,France,FRA,2022,Olivier Giroud
198,199,WC-2022,2022 FIFA Men's World Cup,A-7,Golden Glove,0,P-13162,Martínez,Emiliano,T-03,Argentina,ARG,2022,Emiliano Martínez


## Remove unnecessary columns

In [12]:
data = data.drop(columns=[
    "key_id",
    "tournament_id",
    "tournament_name",
    "award_id",
    "player_id",
    "team_id",
    "team_code",
    "given_name",
    "family_name"
])
data.columns

Index(['award_name', 'shared', 'team_name', 'Year', 'Player Name'], dtype='str')

## Columns rename

In [13]:
columns = {
    "award_name": "Award",
    "shared": "Shared",
    "team_name": "Team"
}

data = data.rename(columns=columns)
data.columns

Index(['Award', 'Shared', 'Team', 'Year', 'Player Name'], dtype='str')

## Convert team names

In [14]:
data["Team"].value_counts()

Team
Brazil            20
Argentina         13
Italy             13
France            12
Germany           11
West Germany      10
Hungary            6
Spain              6
Netherlands        6
England            5
Belgium            5
Uruguay            4
Croatia            4
Peru               3
Poland             3
United States      2
Czechoslovakia     2
Soviet Union       2
Yugoslavia         2
Bulgaria           2
Switzerland        1
Austria            1
Chile              1
Portugal           1
Denmark            1
Cameroon           1
Russia             1
Sweden             1
South Korea        1
Colombia           1
Name: count, dtype: int64

In [15]:
data.loc[data["Team"] == "West Germany", "Team"] = "Germany"
data.loc[data["Team"] == "Soviet Union", "Team"] = "Russia"
data.loc[data["Team"] == "Czechoslovakia", "Team"] = "Czech Republic"
data["Team"].value_counts()

Team
Germany           21
Brazil            20
Argentina         13
Italy             13
France            12
Hungary            6
Spain              6
Netherlands        6
England            5
Belgium            5
Uruguay            4
Croatia            4
Russia             3
Peru               3
Poland             3
United States      2
Czech Republic     2
Yugoslavia         2
Bulgaria           2
Switzerland        1
Austria            1
Chile              1
Portugal           1
Denmark            1
Cameroon           1
Sweden             1
South Korea        1
Colombia           1
Name: count, dtype: int64

# 6. World Cup 2026

In [17]:
golden_boot = {
    "Award": "Golden Boot",
    "Shared": 0,
    "Team": "France",
    "Year": 2026,
    "Player Name": "Kylian Mbappé"
}
silver_boot = {
    "Award": "Silver Boot",
    "Shared": 0,
    "Team": "Argentina",
    "Year": 2026,
    "Player Name": "Lionel Messi"
}
bronze_boot1 = {
    "Award": "Bronze Boot",
    "Shared": 1,
    "Team": "England",
    "Year": 2026,
    "Player Name": "Jude Bellingham"
}
bronze_boot2 = {
    "Award": "Bronze Boot",
    "Shared": 1,
    "Team": "Norway",
    "Year": 2026,
    "Player Name": "Erling Haaland"
}
golden_ball = {
    "Award": "Golden Ball",
    "Shared": 0,
    "Team": "Spain",
    "Year": 2026,
    "Player Name": "Rodri"
}
silver_ball = {
    "Award": "Silver Ball",
    "Shared": 0,
    "Team": "Argentina",
    "Year": 2026,
    "Player Name": "Lionel Messi"
}
bronze_ball = {
    "Award": "Bronze Ball",
    "Shared": 0,
    "Team": "France",
    "Year": 2026,
    "Player Name": "Kylian Mbappé"
}
golden_glove = {
    "Award": "Golden Glove",
    "Shared": 0,
    "Team": "Spain",
    "Year": 2026,
    "Player Name": "Unai Simón"
}
best_young_player = {
    "Award": "Best Young Player",
    "Shared": 0,
    "Team": "Spain",
    "Year": 2026,
    "Player Name": "Pau Cubarsí"
}


wc_2026 = pd.DataFrame([
    golden_ball,
    silver_ball,
    bronze_ball,
    golden_boot,
    silver_boot,
    bronze_boot1,
    bronze_boot2,
    golden_glove,
    best_young_player
])
wc_2026

,Award,Shared,Team,Year,Player Name
0,Golden Ball,0,Spain,2026,Rodri
1,Silver Ball,0,Argentina,2026,Lionel Messi
2,Bronze Ball,0,France,2026,Kylian Mbappé
3,Golden Boot,0,France,2026,Kylian Mbappé
4,Silver Boot,0,Argentina,2026,Lionel Messi
5,Bronze Boot,1,England,2026,Jude Bellingham
6,Bronze Boot,1,Norway,2026,Erling Haaland
7,Golden Glove,0,Spain,2026,Unai Simón
8,Best Young Player,0,Spain,2026,Pau Cubarsí


In [18]:
data = pd.concat([data, wc_2026], ignore_index=True)
data

,Award,Shared,Team,Year,Player Name
0,Golden Boot,0,Argentina,1930,Guillermo Stábile
1,Silver Boot,0,Uruguay,1930,Pedro Cea
2,Bronze Boot,0,United States,1930,Bert Patenaude
3,Golden Boot,0,Czech Republic,1934,Oldřich Nejedlý
4,Silver Boot,1,Germany,1934,Edmund Conen
...,...,...,...,...,...
145,Silver Boot,0,Argentina,2026,Lionel Messi
146,Bronze Boot,1,England,2026,Jude Bellingham
147,Bronze Boot,1,Norway,2026,Erling Haaland
148,Golden Glove,0,Spain,2026,Unai Simón


# 7. Export

In [19]:
df_export = data.reindex(columns=[
    "Year",
    "Award",
    "Player Name",
    "Team",
    "Shared"
])

df_export.head()

,Year,Award,Player Name,Team,Shared
0,1930,Golden Boot,Guillermo Stábile,Argentina,0
1,1930,Silver Boot,Pedro Cea,Uruguay,0
2,1930,Bronze Boot,Bert Patenaude,United States,0
3,1934,Golden Boot,Oldřich Nejedlý,Czech Republic,0
4,1934,Silver Boot,Edmund Conen,Germany,1


In [20]:
df_export.to_csv("../data/processed/awards.csv", index=False)